# 3.0 — Modelado y Evaluación (Fase 3)

**Proyecto**: Predicción del resultado de partidos de fútbol (`FTResult` H/D/A) — Minería de Datos CC442.
**Responsable**: Sergio Pezo (modelado) · Luis Trujillo (evaluación, segunda mitad).

Este cuaderno entrena y compara **6 clasificadores** — Logistic Regression, SVM (LinearSVC),
Random Forest, XGBoost, LightGBM y CatBoost — sobre los datos ya procesados en la Fase 2.

> **Estado**: ESQUELETO plug-and-play. Consume `data/processed/` (salida de `2.0-preprocessing.ipynb`).
> Cuando Arbués publique los datos procesados, este cuaderno corre de inicio a fin.
> Los supuestos sobre el formato de los datos están marcados con `# SUPUESTO:` — ajustar si cambia.

### Convención de 4 bloques (rúbrica Colab)
Cada sección alterna: **explicación teórica → código → interpretación → discusión de alternativas**.


## 1. Configuración del entorno y reproducibilidad

**Reproducibilidad científica absoluta** (rúbrica §11.1): se fija `SEED=42` en `random`, `numpy` y en
cada modelo. Sin esto, dos ejecuciones darían resultados distintos y los experimentos no serían verificables.

In [1]:
import os, random, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# --- Presupuesto de tiempo por modelo (regla del equipo) ---
# Si un modelo tarda mas de MAX_MINUTES_PER_MODEL en un ajuste de prueba,
# se reentrena sobre una submuestra estratificada para no bloquear la noche.
MAX_MINUTES_PER_MODEL = 8
SUBSAMPLE_FRAC = 0.30          # fraccion a usar si un modelo resulta lento
USE_GPU = True                 # boosting en GPU (XGBoost/LightGBM/CatBoost) con fallback a CPU

# --- Rutas del repo ---
ROOT = Path.cwd()
while ROOT.name and not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
for d in (MODELS, RESULTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("data/processed existe:", DATA.exists())

ROOT: /home/sergi/Documents/semesters/eight/dm/pc5/football-prediction
data/processed existe: True


### Disponibilidad de librerías

Los boosting modernos (LightGBM, CatBoost) y Optuna pueden no estar instalados. En lugar de romper el
`Run All`, se detecta su disponibilidad y se **omite** el modelo faltante con un aviso. Para instalarlos:

```bash
pip install lightgbm catboost optuna
```

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

HAS = {}
try:
    from xgboost import XGBClassifier; HAS["xgboost"] = True
except ImportError: HAS["xgboost"] = False
try:
    from lightgbm import LGBMClassifier; HAS["lightgbm"] = True
except ImportError: HAS["lightgbm"] = False
try:
    from catboost import CatBoostClassifier; HAS["catboost"] = True
except ImportError: HAS["catboost"] = False

print("Librerias de boosting disponibles:")
for k, v in HAS.items():
    print(f"  {k:10s}: {'OK' if v else 'FALTA (pip install '+k+')'}")

Librerias de boosting disponibles:
  xgboost   : OK
  lightgbm  : FALTA (pip install lightgbm)
  catboost  : FALTA (pip install catboost)


## 2. Carga de datos procesados

**Supuesto de interfaz con Fase 2**: el notebook de preprocessing guarda train/val/test ya
transformados (escalados, codificados, sin fuga de datos). Se intenta primero `.parquet`
(preserva nombres de columnas, útil para SHAP y feature importance) y luego `.npy`.

Si Arbués usa otro formato/nombres, ajustar SOLO esta celda.

In [3]:
def load_split(name):
    """Carga un split (X_name, y_name) desde parquet o npy. Devuelve (X, y)."""
    pq_X, pq_y = DATA / f"X_{name}.parquet", DATA / f"y_{name}.parquet"
    np_X, np_y = DATA / f"X_{name}.npy", DATA / f"y_{name}.npy"
    if pq_X.exists():
        X = pd.read_parquet(pq_X)
        y = pd.read_parquet(pq_y).squeeze("columns")
        return X, y
    if np_X.exists():
        return np.load(np_X), np.load(np_y, allow_pickle=True)
    raise FileNotFoundError(
        f"No se encontraron X_{name}/y_{name} en {DATA}. "
        "Ejecuta primero 2.0-preprocessing.ipynb (Arbues, Fase 2).")

if DATA.exists() and any(DATA.glob("X_train.*")):
    X_train, y_train = load_split("train")
    X_val,   y_val   = load_split("val")
    X_test,  y_test  = load_split("test")
    print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
    print("Clases (train):", pd.Series(y_train).value_counts(normalize=True).round(3).to_dict())
    DATA_READY = True
else:
    print("⚠️  Aun no hay datos procesados. El resto del notebook esta listo para cuando existan.")
    print("    Esqueleto validado; esperando la salida de la Fase 2 (Arbues).")
    DATA_READY = False

⚠️  Aun no hay datos procesados. El resto del notebook esta listo para cuando existan.
    Esqueleto validado; esperando la salida de la Fase 2 (Arbues).


## 3. Baseline: clasificador ingenuo (DummyClassifier)

**Por qué es obligatorio** (rúbrica §12, matriz de errores): en un dataset desbalanceado, un clasificador
que siempre vota la clase mayoritaria (H ≈ 44.6%) ya obtiene ~45% de accuracy sin aprender nada. Todo
modelo debe **superar claramente este piso**; si no, no aporta. Reportamos accuracy y F1-weighted del
baseline como referencia inferior.

In [4]:
from sklearn.metrics import accuracy_score, f1_score

if DATA_READY:
    dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
    dummy.fit(X_train, y_train)
    pred = dummy.predict(X_val)
    print(f"Baseline (most_frequent) — Acc(val): {accuracy_score(y_val, pred):.4f} | "
          f"F1w(val): {f1_score(y_val, pred, average='weighted'):.4f}")

## 4. Definición de modelos y espacios de búsqueda

Se separan los modelos en dos grupos según su costo:
- **Ligeros** (LogReg, LinearSVC) → `GridSearchCV` exhaustivo.
- **Pesados** (RF, XGBoost, LightGBM, CatBoost) → `RandomizedSearchCV` (más eficiente en presupuesto).

Todos con `Stratified K-Fold (5)` y **`F1-weighted`** como métrica de optimización (adecuada al multiclase
desbalanceado). Los grids son moderados a propósito para respetar el presupuesto de 8 min/modelo; se pueden
ampliar si sobra tiempo.

In [5]:
from scipy.stats import randint, uniform

def gpu_kwargs(lib):
    """Parametros GPU por libreria (con fallback silencioso a CPU si USE_GPU=False)."""
    if not USE_GPU:
        return {}
    return {
        "xgboost":  dict(tree_method="hist", device="cuda"),
        "lightgbm": dict(device="gpu"),
        "catboost": dict(task_type="GPU"),
    }.get(lib, {})

# (estimador, param_grid, tipo_busqueda). Se construye solo con lo disponible.
def build_registry():
    reg = {}
    reg["LogisticRegression"] = (
        LogisticRegression(max_iter=2000, random_state=SEED, class_weight="balanced"),
        {"C": [0.01, 0.1, 1, 10], "penalty": ["l2"], "solver": ["lbfgs"]},
        "grid")
    reg["LinearSVC"] = (
        LinearSVC(random_state=SEED, class_weight="balanced", dual="auto"),
        {"C": [0.01, 0.1, 1, 10]},
        "grid")
    reg["RandomForest"] = (
        RandomForestClassifier(random_state=SEED, class_weight="balanced", n_jobs=-1),
        {"n_estimators": randint(200, 600), "max_depth": randint(6, 30),
         "min_samples_split": randint(2, 20), "max_features": ["sqrt", "log2"]},
        "random")
    if HAS["xgboost"]:
        reg["XGBoost"] = (
            XGBClassifier(random_state=SEED, eval_metric="mlogloss",
                          n_jobs=-1, **gpu_kwargs("xgboost")),
            {"n_estimators": randint(200, 500), "learning_rate": uniform(0.01, 0.29),
             "max_depth": randint(3, 10), "subsample": uniform(0.6, 0.4),
             "colsample_bytree": uniform(0.6, 0.4), "gamma": uniform(0, 5)},
            "random")
    if HAS["lightgbm"]:
        reg["LightGBM"] = (
            LGBMClassifier(random_state=SEED, class_weight="balanced",
                           n_jobs=-1, verbose=-1, **gpu_kwargs("lightgbm")),
            {"n_estimators": randint(200, 500), "learning_rate": uniform(0.01, 0.29),
             "num_leaves": randint(20, 150), "subsample": uniform(0.6, 0.4)},
            "random")
    if HAS["catboost"]:
        reg["CatBoost"] = (
            CatBoostClassifier(random_seed=SEED, verbose=0, **gpu_kwargs("catboost")),
            {"iterations": randint(200, 600), "learning_rate": uniform(0.01, 0.29),
             "depth": randint(4, 10)},
            "random")
    return reg

REGISTRY = build_registry()
print("Modelos a entrenar:", list(REGISTRY.keys()))

Modelos a entrenar: ['LogisticRegression', 'LinearSVC', 'RandomForest', 'XGBoost']


### Discusión de alternativas (bloque de rúbrica)

- **LinearSVC vs SVC(kernel rbf)**: SVC con kernel no lineal es O(n²) — inviable en 226k filas. LinearSVC
  escala, a costa de no dar probabilidades nativas (relevante para ROC-AUC/log-loss en Fase 4: usar
  `CalibratedClassifierCV` o `decision_function`).
- **F1-weighted vs macro**: weighted pondera por soporte (refleja el rendimiento global); macro trata las 3
  clases por igual (penaliza más el mal desempeño en empates). Se optimiza weighted, pero en Fase 4 se
  reportan ambas para exponer la debilidad esperada en la clase D.
- **class_weight="balanced"**: compensa el desbalance sin remuestrear. Alternativa: SMOTE en train (Fase 2).

## 5. Bucle de entrenamiento con presupuesto de tiempo

Para cada modelo: (1) un **ajuste de prueba** mide el costo de un solo `fit`; (2) si excede el presupuesto,
se entrena la búsqueda sobre una **submuestra estratificada** (`SUBSAMPLE_FRAC`); (3) se corre la búsqueda de
hiperparámetros con CV estratificada; (4) se registran best params, tiempo y métricas en validación; (5) se
serializa el mejor estimador en `models/`.

In [6]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split
import joblib

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
N_ITER = 20  # iteraciones para RandomizedSearchCV

def preflight_fit_seconds(estimator, X, y, sample=5000):
    """Estima el costo de un fit con una submuestra pequena."""
    n = min(sample, len(X))
    Xs = X.iloc[:n] if hasattr(X, "iloc") else X[:n]
    ys = y.iloc[:n] if hasattr(y, "iloc") else y[:n]
    t0 = time.time()
    est = estimator.__class__(**estimator.get_params()); est.fit(Xs, ys)
    return (time.time() - t0), n

def stratified_subsample(X, y, frac):
    Xr, _, yr, _ = train_test_split(X, y, train_size=frac, stratify=y, random_state=SEED)
    return Xr, yr

def train_all():
    rows = []
    for name, (est, grid, kind) in REGISTRY.items():
        print(f"\n=== {name} ===")
        Xtr, ytr = X_train, y_train
        # --- presupuesto de tiempo ---
        secs, n = preflight_fit_seconds(est, X_train, y_train)
        est_full = secs * (len(X_train) / n)  # extrapolacion lineal aproximada
        print(f"  fit de prueba: {secs:.1f}s sobre {n} filas -> ~{est_full/60:.1f} min/fit en full")
        if est_full / 60 > MAX_MINUTES_PER_MODEL:
            Xtr, ytr = stratified_subsample(X_train, y_train, SUBSAMPLE_FRAC)
            print(f"  ⏱️  supera {MAX_MINUTES_PER_MODEL} min -> submuestra {SUBSAMPLE_FRAC:.0%} ({len(Xtr)} filas)")

        search_cls = GridSearchCV if kind == "grid" else RandomizedSearchCV
        kw = dict(scoring="f1_weighted", cv=CV, n_jobs=-1, refit=True)
        if kind == "random":
            kw.update(n_iter=N_ITER, random_state=SEED)
        search = search_cls(est, grid, **kw)

        t0 = time.time(); search.fit(Xtr, ytr); dt = time.time() - t0
        best = search.best_estimator_
        pval_pred = best.predict(X_val)
        acc = accuracy_score(y_val, pval := pval_pred)
        f1w = f1_score(y_val, pval, average="weighted")
        f1m = f1_score(y_val, pval, average="macro")
        print(f"  best CV f1w: {search.best_score_:.4f} | val Acc: {acc:.4f} | "
              f"val F1w: {f1w:.4f} | F1m: {f1m:.4f} | {dt:.0f}s")

        joblib.dump(best, MODELS / f"{name}.pkl")
        rows.append(dict(model=name, best_params=json.dumps(search.best_params_, default=str),
                         cv_f1w=round(search.best_score_, 4), val_acc=round(acc, 4),
                         val_f1w=round(f1w, 4), val_f1m=round(f1m, 4),
                         train_seconds=round(dt, 1), n_train=len(Xtr)))
    return pd.DataFrame(rows).sort_values("val_f1w", ascending=False)

if DATA_READY:
    results_df = train_all()
    results_df.to_csv(RESULTS / "grid_search_results.csv", index=False)
    display(results_df)
else:
    print("Esperando datos procesados. La funcion train_all() esta lista para ejecutarse.")

Esperando datos procesados. La funcion train_all() esta lista para ejecutarse.


## 6. Registro de resultados (validación)

La tabla siguiente resume el desempeño en **validación** de cada modelo. La **evaluación final en test**
(Precision/Recall/F1 por clase, Cohen's Kappa, ROC-AUC OvR, log-loss, matrices de confusión, Wilcoxon y
SHAP) se realiza en la **segunda mitad de este notebook — Fase 4, responsable Luis**.

> Mejor modelo por `val_f1w` → se usa como candidato para el análisis profundo de Fase 4.

In [7]:
if DATA_READY:
    best_model_name = results_df.iloc[0]["model"]
    print("Mejor modelo (val F1w):", best_model_name)
    print("Modelos serializados en:", MODELS)
    print("Resultados en:", RESULTS / "grid_search_results.csv")
else:
    print("Pendiente de datos procesados.")

Pendiente de datos procesados.


---
### Handoff a Fase 4 (Luis)

Cuando esta primera mitad corra con datos reales, quedan disponibles:
- `models/*.pkl` — un estimador entrenado por modelo.
- `results/grid_search_results.csv` — tabla de métricas en validación.

Fase 4 continúa **debajo de esta celda**: métricas en test, curvas ROC/PR, matriz de confusión,
Wilcoxon signed-rank (mejor modelo vs baselines), SHAP y permutation importance. Figuras → `figures/`.